# 04 — Huấn luyện mô hình (Modeling)

Thử nghiệm dự đoán thời gian điều trị: hồi quy tuyến tính, phân loại nhị phân (Ngắn/Dài) và 3 lớp (Ngắn/Trung bình/Dài) có cân bằng lớp bằng SMOTE. **Đầu vào**: `data/processed/model_df.pkl`.

> ⚠️ Các cell thử nghiệm bên dưới **thay đổi `model_df` tại chỗ** (đổi cột `Thời gian điều trị` sang nhãn). Nếu muốn chạy lại một thử nghiệm khác, hãy **chạy lại cell nạp dữ liệu** để lấy `model_df` gốc trước.

### Import & nạp dữ liệu

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import config

In [ ]:
model_df = pd.read_pickle(config.MODEL_DF_PKL)
model_df.head()

### Import thư viện mô hình hóa

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
import math
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import mean_squared_error

### Thử nghiệm 1 — Hồi quy tuyến tính dự đoán số ngày điều trị

In [ ]:
def training():
  df = model_df.copy()
  #df['Thời gian điều trị'] = list(map(lambda x: 'Ngắn' if (x <= 7) else 'Dài',df['Thời gian điều trị']))
  #df = df.iloc[:, :-22]
  df = df.drop(columns = ['DANTOC','TONGCP','Vùng miền','Tháng nhập viện'])
  df['Thời gian điều trị'] = list(map(lambda x: math.ceil(x),df['Thời gian điều trị']))
  df = df[df['Thời gian điều trị'] <= 60]
  X = df.drop(columns = ['Thời gian điều trị'])
  y = df['Thời gian điều trị']
  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)
  scaler = StandardScaler()
  X_train = scaler.fit_transform(X_train)
  X_test = scaler.transform(X_test)
  model = LinearRegression()
  model.fit(X_train, y_train)
  print(model.score(X_test, y_test))
  print(mean_squared_error(y_test, model.predict(X_test)))
  #model = LogisticRegression()
  # model = RandomForestClassifier(random_state = 42)
  # model.fit(X_train, y_train)
  # print(classification_report(y_test, model.predict(X_test)))
  # print(confusion_matrix(y_test, model.predict(X_test)))
  # sns.boxplot(data = df[df['Thời gian điều trị'] <= 300], x = 'Thời gian điều trị')
  # plt.show()
  return df
training()

0.16897835989918775
43.710295504891896


,Tuổi,Thời gian điều trị,Kinh độ,Vĩ độ,Số lượng bệnh/hỗ trợ y tế,Số lượng danh mục bệnh I,Số lượng danh mục bệnh II,Số lượng danh mục bệnh III,Số lượng danh mục bệnh IV,Số lượng danh mục bệnh V,...,Số lượng danh mục bệnh XIII,Số lượng danh mục bệnh XIV,Số lượng danh mục bệnh XV,Số lượng danh mục bệnh XVI,Số lượng danh mục bệnh XVII,Số lượng danh mục bệnh XVIII,Số lượng danh mục bệnh XIX,Số lượng danh mục bệnh XX,Số lượng danh mục bệnh XXI,Số lượng danh mục bệnh XXII
0,28,1,109.191606,13.793053,1,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
1,18,1,109.181311,13.794558,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,36,1,109.068601,13.878986,2,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
3,5,3,109.148271,13.786352,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,5,7,109.181311,13.794558,2,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
68757,73,31,109.148271,13.786352,2,0,0,1,0,0,...,0,0,0,0,0,0,0,0,1,0
68758,24,43,109.101382,13.887191,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
68759,42,43,109.030093,13.906113,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
68760,40,52,109.148393,14.220117,2,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0


### Thử nghiệm 2 — Phân loại nhị phân (Ngắn ≤10 ngày / Dài)

In [ ]:



model_df['Thời gian điều trị'] = list(map(lambda x: 'Ngắn' if (x <= 10) else 'Dài',model_df['Thời gian điều trị']))
X = model_df.drop(columns = ['Thời gian điều trị','TONGCP'])
X = pd.get_dummies(X)
y = model_df['Thời gian điều trị']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)
print(y_train.value_counts())


Thời gian điều trị
Ngắn    43856
Dài     10433
Name: count, dtype: int64


In [ ]:
# sm = SMOTE(sampling_strategy = {'Trung bình': 20000, 'Dài': 4000}, random_state = 42)
# X_train, y_train = sm.fit_resample(X_train, y_train)
model = RandomForestClassifier(n_estimators = 300, bootstrap = False, max_depth = 10, min_samples_split = 1000, random_state = 42)
model.fit(X_train, y_train)

RandomForestClassifier(bootstrap=False, max_depth=10, min_samples_split=1000,
                       n_estimators=300, random_state=42)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(y_test, model.predict(X_test)))
print(confusion_matrix(y_test, model.predict(X_test)))

              precision    recall  f1-score   support

         Dài       0.75      0.13      0.22      2644
        Ngắn       0.82      0.99      0.90     10929

    accuracy                           0.82     13573
   macro avg       0.79      0.56      0.56     13573
weighted avg       0.81      0.82      0.77     13573

[[  338  2306]
 [  113 10816]]


### Thử nghiệm 3 — Phân loại 3 lớp + SMOTE, tinh chỉnh ngưỡng & độ quan trọng đặc trưng

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE


model_df['Thời gian điều trị'] = list(map(lambda x: 'Ngắn' if (x <= 10) else
                                                    'Trung bình' if (x <= 60) else 'Dài',model_df['Thời gian điều trị']))
X = model_df.drop(columns = ['Thời gian điều trị','TONGCP'])
X = pd.get_dummies(X)
y = model_df['Thời gian điều trị']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)
print(y_train.value_counts())
sm = SMOTE(sampling_strategy = {'Trung bình': 20000, 'Dài': 4000}, random_state = 42)
X_train, y_train = sm.fit_resample(X_train, y_train)
model = RandomForestClassifier(n_estimators = 300, bootstrap = False, max_depth = 15, min_samples_split = 100, random_state = 42)
model.fit(X_train, y_train)

Thời gian điều trị
Ngắn          43856
Trung bình     9692
Dài             741
Name: count, dtype: int64


RandomForestClassifier(bootstrap=False, max_depth=15, min_samples_split=100,
                       n_estimators=300, random_state=42)

In [ ]:
def threshold():
  prob_df = pd.DataFrame(model.predict_proba(X_test), columns = ['Dài','Ngắn','Trung bình'])
  prob_df['Prediction'] = list(map(lambda x,y,z: 'Dài' if (x == max(x,y,z)) else
                                                 'Ngắn' if (y > z and y - z > 0.15) else
                                                 'Trung bình',prob_df['Dài'],prob_df['Ngắn'],prob_df['Trung bình']))
  prob_df['Actual'] = y_test.reset_index(drop = True)
  print(classification_report(prob_df['Actual'], prob_df['Prediction']))
  print(confusion_matrix(prob_df['Actual'], prob_df['Prediction']))

threshold()


              precision    recall  f1-score   support

         Dài       0.56      0.78      0.65       175
        Ngắn       0.88      0.83      0.86     10929
  Trung bình       0.40      0.48      0.43      2469

    accuracy                           0.77     13573
   macro avg       0.61      0.70      0.65     13573
weighted avg       0.79      0.77      0.78     13573

[[ 137    8   30]
 [  60 9106 1763]
 [  49 1243 1177]]


In [ ]:
def importance():
  feature_importances = model.feature_importances_
  feature_names = X.columns
  feature_importance_df = pd.DataFrame({
      'Feature': feature_names,
      'Importance': feature_importances
  })
  feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)
  print(feature_importance_df)

importance()

                                    Feature  Importance
25               Số lượng danh mục bệnh XXI    0.317982
0                                      Tuổi    0.171586
19                Số lượng danh mục bệnh XV    0.062457
3                                     Vĩ độ    0.061536
4                 Số lượng bệnh/hỗ trợ y tế    0.058180
2                                   Kinh độ    0.057972
1                           Tháng nhập viện    0.044761
5                  Số lượng danh mục bệnh I    0.033937
6                 Số lượng danh mục bệnh II    0.032736
13                Số lượng danh mục bệnh IX    0.023378
15                Số lượng danh mục bệnh XI    0.021797
22             Số lượng danh mục bệnh XVIII    0.015048
17              Số lượng danh mục bệnh XIII    0.014138
23               Số lượng danh mục bệnh XIX    0.013028
12              Số lượng danh mục bệnh VIII    0.010655
11               Số lượng danh mục bệnh VII    0.009645
14                 Số lượng danh mục bệnh X    0